# DEMO TỪNG HÀM / METHOD LANGCHAIN PDF QA

Notebook dùng **chính các hàm trong ZIP mới**, PDF mẫu kèm theo. Chạy lần lượt `Shift+Enter`. Cần cài đặt bằng `01_CAI_DAT_LAN_DAU.bat`. **Kết quả cần được chạy thật trên máy bạn, không có dữ liệu benchmark bịa sẵn.**

In [ ]:
from pathlib import Path
import os, sys, time
# Khi mở từ 03_MO_NOTEBOOK.bat, JupyterLab chạy ở thư mục dự án.
ROOT = Path.cwd()
if not (ROOT / 'config.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Thư mục dự án:', ROOT)

## Method 1. `list_pdfs()` — Input: thư mục / Output: danh sách PDF

In [ ]:
from config import PDF_DIR, INDEX_DIR, EMBEDDING_MODEL, LLM_MODEL
from src.pdf_reader import list_pdfs, load_pdfs
pdfs = list_pdfs(PDF_DIR)
print([p.name for p in pdfs])

## Method 2. `load_pdfs()` và `PyPDFLoader.load()` — Input: các PDF / Output: Document theo trang

In [ ]:
pages = load_pdfs(PDF_DIR)
print('Số trang có chữ:', len(pages))
print('Metadata:', pages[0].metadata)
print('Nội dung mẫu:', pages[0].page_content[:400])

## Verify the supplied PDF (ground truth before model evaluation)

The book-club sample PDF is fictional. We check known facts directly from the text so retrieval has an objective reference. This test does **not** run the LLM.


In [ ]:
reference_facts = {
    'founded_in_2024': ('2024' in pages[0].page_content),
    'meets_on_saturday': ('thứ Bảy' in pages[0].page_content),
    'borrowing_14_days': ('14 ngày' in pages[1].page_content),
    'two_books': ('2 cuốn' in pages[1].page_content),
    'does_not_define_langchain': not any('LangChain là một công cụ' in p.page_content for p in pages),
}
for label, result in reference_facts.items(): print(label, ':', 'PASS' if result else 'CHECK SOURCE')
assert all(reference_facts.values()), 'Reference PDF was changed; update the verification statements.'


## Method 3. `split_documents()` — Input: Document + tham số / Output: chunks

In [ ]:
from src.text_chunker import split_documents
chunks = split_documents(pages, chunk_size=900, chunk_overlap=150)
print('Số chunks:', len(chunks))
print('Chunk đầu:', chunks[0].page_content[:400])

## Method 4. `make_embeddings()` — Input: tên model / Output: vector hoá đa ngôn ngữ

In [ ]:
from src.vector_store import make_embeddings
emb = make_embeddings(EMBEDDING_MODEL)
vec = emb.embed_query('Câu lạc bộ họp vào thứ mấy?')
print('Chiều vector:', len(vec), '5 giá trị đầu:', vec[:5])

## Method 5. `build_vector_store()` — Input: chunks + embedding / Output: FAISS

In [ ]:
from src.vector_store import build_vector_store, save_vector_store, load_vector_store
store = build_vector_store(chunks, emb)
print('Số vector:', store.index.ntotal)

## Method 6. `save_vector_store()` + `load_vector_store()` — Input: FAISS / Output: index lưu & nạp

In [ ]:
save_vector_store(store, INDEX_DIR)
loaded_store = load_vector_store(INDEX_DIR, emb)
print('Số vector đã nạp lại:', loaded_store.index.ntotal)

## Method 7. `retrieve_documents()` — Input: câu hỏi / Output: top-k Document

In [ ]:
from src.qa import retrieve_documents, format_context
question = 'Câu lạc bộ họp vào thứ mấy, mấy giờ?'
retrieved = retrieve_documents(loaded_store, question, k=2)
for i, doc in enumerate(retrieved,1):
    print(i, doc.metadata.get('source'), 'trang', doc.metadata.get('page',0)+1, doc.page_content[:180])

## Method 8. `format_context()` — Input: chunks / Output: chuỗi context

In [ ]:
context = format_context(retrieved)
print(context[:1000])

## Method 9. `generate_answer()` — Input: câu hỏi + context / Output: câu trả lời. Cần Ollama đang chạy.

In [ ]:
from src.qa import generate_answer, answer_question
from app import check_ollama
check_ollama(LLM_MODEL)
answer = generate_answer(question, retrieved, LLM_MODEL)
print(answer)

## Method 10. `answer_question()` — Input: store, question, model, k / Output: dict có answer, sources

In [ ]:
t0=time.perf_counter()
result=answer_question(loaded_store, question, LLM_MODEL, k=4)
print('Đáp án:', result['answer'])
print('Nguồn:', result['sources'])
print('Thời gian:', round(time.perf_counter()-t0,2),'giây')

## Method 11. `prepare_store()` — Index tự động khi thay đổi PDF / cấu hình

In [ ]:
from src.pipeline import pdf_signature, build_manifest, prepare_store
print('Dấu hiệu PDF:', pdf_signature(PDF_DIR))
print('Manifest:', build_manifest(PDF_DIR))
store2, created_new, num_pages, num_chunks = prepare_store(PDF_DIR, INDEX_DIR)
print('Tạo mới:',created_new,'trang:',num_pages,'chunks:',num_chunks)

## Method I/O inventory: 17 project-level functions

These methods are defined in the supplied Python modules. Full real source code, example calls, and I/O contracts are in `tai_lieu/05_METHOD_INPUT_OUTPUT_CODE_EN.pdf`.


In [ ]:
import inspect
from src import pdf_reader, text_chunker, vector_store, qa, pipeline
import app
function_groups = {
    'src/pdf_reader.py': [pdf_reader.list_pdfs,pdf_reader.load_pdfs],
    'src/text_chunker.py': [text_chunker.split_documents],
    'src/vector_store.py': [vector_store.make_embeddings,vector_store.build_vector_store,vector_store.save_vector_store,vector_store.load_vector_store],
    'src/qa.py': [qa.retrieve_documents,qa.format_context,qa.generate_answer,qa.answer_question],
    'src/pipeline.py': [pipeline.pdf_signature,pipeline.build_manifest,pipeline.prepare_store],
    'app.py': [app.check_ollama,app.save_history,app.main],
}
assert sum(len(group) for group in function_groups.values()) == 17
for module, functions in function_groups.items():
    for func in functions:
        print(f'{module} | {func.__name__}{inspect.signature(func)}')


## Five real experiments A–E (change chunk_size, overlap, top-k)

All five experiments use the same sample PDF and question. The expected answer is **14 days on page 2**. We measure separate indexing and retrieval times, and record whether page 2 is retrieved. This is **not a full accuracy or LLM benchmark**. Values are produced only after executing this notebook on the actual machine.


In [ ]:
# Real measurements are produced ONLY when this cell runs on a configured machine.
# Each setting rebuilds a temporary vector store in memory, not the main saved index.
configurations = [
    {'name':'A','chunk_size':500,'overlap':50,'k':2},
    {'name':'B','chunk_size':700,'overlap':100,'k':3},
    {'name':'C','chunk_size':900,'overlap':150,'k':4},
    {'name':'D','chunk_size':1100,'overlap':200,'k':3},
    {'name':'E','chunk_size':1300,'overlap':250,'k':5},
]
question = 'Thời hạn mượn sách là bao nhiêu ngày?'
experiment_results = []
for cfg in configurations:
    start = time.perf_counter()
    candidate_chunks = split_documents(pages,cfg['chunk_size'],cfg['overlap'])
    candidate_store = build_vector_store(candidate_chunks,emb)
    indexing_s = time.perf_counter()-start
    start = time.perf_counter()
    hits = retrieve_documents(candidate_store,question,cfg['k'])
    retrieval_s = time.perf_counter()-start
    pages_hit = [int(d.metadata.get('page',0))+1 for d in hits]
    row = {**cfg,
           'num_chunks':len(candidate_chunks),
           'num_retrieved':len(hits),
           'index_seconds':round(indexing_s,4),
           'retrieval_seconds':round(retrieval_s,4),
           'retrieved_pages':';'.join(map(str,pages_hit)),
           'top1_page_2': int(bool(pages_hit and pages_hit[0]==2)),
           'has_page_2_in_top_k':int(2 in pages_hit),
           'question':question}
    experiment_results.append(row)
for row in experiment_results:
    print(row)
print('IMPORTANT: top1_page_2 is a reference-page check for THIS one question, not overall answer accuracy.')


## Export actual experiment data

The CSV is written **only after the previous cell finishes**. Values must not be copied or invented in advance.


In [ ]:
import csv
OUT=ROOT/'ket_qua'
OUT.mkdir(parents=True,exist_ok=True)
with (OUT/'ket_qua_5_cau_hinh.csv').open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=experiment_results[0].keys())
    writer.writeheader();writer.writerows(experiment_results)
print('Đã lưu:', OUT/'ket_qua_5_cau_hinh.csv')

## Kiểm tra tình huống trả lời ngoài PDF

Ví dụ: “Ai là hiệu trưởng?” PDF mẫu không nêu. LLM cần từ chối khẳng định không có nguồn.

In [ ]:
outside = answer_question(loaded_store, 'Ai là hiệu trưởng trong tài liệu này?', LLM_MODEL, k=4)
print(outside['answer'])
print('Nguồn được truy xuất:', outside['sources'])